# Structured Output

Models can be requested to provide their response in a format matching a given schema. this is useful for ensuring the output can be easily parsed and used in subsequent processing. LangChain supports multiple schema types and methods for enforcing structured output.

## Pydantic

Pydantic models provide the richest feature set with field validation, descriptions, and nested structures

In [1]:
import os
from dotenv import load_dotenv
from langchain.chat_models import  init_chat_model
load_dotenv()

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
model = init_chat_model("groq:qwen/qwen3-32b")
model

ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x7c5454d95fd0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x7c5454d96a50>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [2]:
from pydantic import BaseModel, Field

class Movide(BaseModel):
    title: str = Field(..., description="The title of the movie")
    director: str = Field(..., description="The director of the movie")
    release_year: int = Field(..., description="The year the movie was released")
    rating: float = Field(..., description="The rating of the movie out of 10")

In [3]:
model_with_structured_output = model.with_structured_output(Movide)
model_with_structured_output

_ChatModelBinding(bound=ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x7c5454d95fd0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x7c5454d96a50>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movide', 'description': '', 'parameters': {'properties': {'title': {'description': 'The title of the movie', 'type': 'string'}, 'director': {'description': 'The director of the movie', 'type': 'string'}, 'release_year': {'description': 'The year the movie was released', 'type': 'integer'}, 'rating': {'description': 'The rating of the mov

In [4]:
model.invoke("Provide details about the movie Inception")

AIMessage(content='<think>\nOkay, so I need to provide details about the movie Inception. Let me start by recalling what I know about it. It\'s a science fiction action film directed by Christopher Nolan, right? The lead actor is Leonardo DiCaprio. The main character\'s name is Dom Cobb, and he\'s a thief who can enter people\'s dreams to steal information. That\'s the basic premise.\n\nI remember that the concept of Inception is about planting an idea into someone\'s subconscious, which is called "inception." The antagonist is someone named Mal, who is Cobb\'s wife. She\'s part of the dream world and causes some issues because she\'s stuck in the dream. There\'s also a character named Arthur, played by Joseph Gordon-Levitt, who\'s Cobb\'s partner. He\'s the one who does the fighting in the dream world. And then there\'s Eames, played by Tom Hardy, who can mimic people\'s identities. \n\nThe movie has this layered structure where they go into different levels of dreams. Each level has 

In [5]:
model_with_structured_output.invoke("Provide details about the movie Inception")

Movide(title='Inception', director='Christopher Nolan', release_year=2010, rating=8.8)

## Message output alongside parsed structure

In [7]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    """A movie with its details"""
    title: str = Field(..., description="The title of the movie")
    director: str = Field(..., description="The director of the movie")
    release_year: int = Field(..., description="The year the movie was released")
    rating: float = Field(..., description="The rating of the movie out of 10")

model_with_structured_output = model.with_structured_output(Movie, include_raw=True)
response = model_with_structured_output.invoke("Provide details about the movie Inception")
print(response)

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': "Okay, the user is asking for details about the movie Inception. Let me check the tools provided. There's a Movie function that requires title, director, release year, and rating. I need to fill in those parameters. I remember Inception was directed by Christopher Nolan. It came out in 2010, I think. The rating is probably around 8.8 on IMDb. Let me confirm the release year and rating to make sure. Yeah, 2010 is correct, and the rating is 8.8. So I'll structure the tool call with those details.\n", 'tool_calls': [{'id': '80e7xnx91', 'function': {'arguments': '{"director":"Christopher Nolan","rating":8.8,"release_year":2010,"title":"Inception"}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 172, 'prompt_tokens': 235, 'total_tokens': 407, 'completion_time': 0.282397514, 'completion_tokens_details': {'reasoning_tokens': 123}, 'prompt_time': 0.019710675, 'prompt_tokens_

## Nested Structure

In [11]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name: str = Field(..., description="The name of the actor")
    role: str = Field(..., description="The role of the actor in the movie")

class MovieDetails(BaseModel):
    """A movie with its details and actors"""
    title: str = Field(..., description="The title of the movie")
    director: str = Field(..., description="The director of the movie")
    release_year: int = Field(..., description="The year the movie was released")
    rating: float = Field(..., description="The rating of the movie out of 10")
    actors: list[Actor] = Field(..., description="A list of actors in the movie")
    generes: list[str] = Field(..., description="A list of genres of the movie")
    budget: float | None = Field(None, description="The budget of the movie in millions")

model_with_structured_output = model.with_structured_output(MovieDetails)

response = model_with_structured_output.invoke("Provide details about the movie Inception")
print(response)

title='Inception' director='Christopher Nolan' release_year=2010 rating=8.8 actors=[Actor(name='Leonardo DiCaprio', role='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Ellen Page', role='Ariadne')] generes=['Science Fiction', 'Action', 'Thriller'] budget=160.0
